# Deep Past Challenge - Inference v5 (Simple, Proven)

This notebook is a MINIMAL adaptation of the reference 35.5-scoring notebook.
- Single model on single GPU (no device_map)
- FP32 (no float16 — ByT5 needs full precision)
- MBR with 4 beam + 2 sample = 6 candidates
- chrF++ MBR utility
- BetterTransformer for speed
- Fixed postprocessing (fractions before forbidden char removal)

**Expected score: ~35.5** (matching reference baseline)

### Models to Attach
1. `mattiaangeli/byt5-akkadian-mbr-v2` (REQUIRED — 582M params, scores 35.5)

In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
import os
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'

import re, json, time, warnings, random, math, logging
from pathlib import Path
from typing import List
from contextlib import nullcontext

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import sacrebleu

warnings.filterwarnings('ignore')

# BF16 support check (T4 does NOT support BF16 → stays FP32)
def _bf16_ok():
    if not torch.cuda.is_available(): return False
    try: return bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)())
    except: return False

def amp_ctx(enabled=True):
    if enabled and torch.cuda.is_available() and _bf16_ok():
        return torch.autocast(device_type='cuda', dtype=torch.bfloat16)
    return nullcontext()

USE_BF16 = _bf16_ok()

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}')
print(f'BF16: {USE_BF16}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_mem/1e9:.1f}GB)')

In [ ]:
# ============================================================
# Cell 2: Configuration (matched to reference 35.5 notebook)
# ============================================================

# Model path — mattiaangeli's model FIRST (proven 35.5)
MODEL_PATHS = [
    '/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1',
    '/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1',
    '/kaggle/input/byt5-akkadian-mbr-v2',
    # Fine-tuned model (only if it exists)
    '/kaggle/input/byt5-akkadian-finetuned/final',
    '/kaggle/input/byt5-akkadian-finetuned',
    # Local testing
    '../models/byt5-akkadian/final',
]

MODEL_PATH = None
for p in MODEL_PATHS:
    if Path(p).exists():
        MODEL_PATH = p
        break
assert MODEL_PATH, f'No model found! Tried: {MODEL_PATHS}'

TEST_PATH = '/kaggle/input/deep-past-initiative-machine-translation/test.csv'
if not Path(TEST_PATH).exists():
    TEST_PATH = '../data/deep-past-initiative-machine-translation/test.csv'

OUTPUT_DIR = '/kaggle/working/' if Path('/kaggle').exists() else './output/'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Generation params (EXACTLY matching reference)
MAX_LENGTH = 384
BATCH_SIZE = 4
NUM_WORKERS = 4
NUM_BEAMS = 8
MAX_NEW_TOKENS = 256
LENGTH_PENALTY = 1.3
REPETITION_PENALTY = 1.2

# MBR config (EXACTLY matching reference)
USE_MBR = True
MBR_BEAM_CANDS = 4
MBR_SAMPLE_CANDS = 2
MBR_TOP_P = 0.9
MBR_TEMPERATURE = 0.7

PREFIX = 'translate Akkadian to English: '

print(f'Model: {MODEL_PATH}')
print(f'Test: {TEST_PATH}')
print(f'MBR: {MBR_BEAM_CANDS} beam + {MBR_SAMPLE_CANDS} sample = {MBR_BEAM_CANDS + MBR_SAMPLE_CANDS} candidates')
print(f'Beams: {NUM_BEAMS}, length_penalty: {LENGTH_PENALTY}, batch: {BATCH_SIZE}')

In [ ]:
# ============================================================
# Cell 3: Preprocessing (minimal — matched to model training)
# ============================================================
_RE_BIG = re.compile(r'(\.{3,}|\u2026+|\u2026\u2026)')
_RE_SMALL = re.compile(r'(xx+|\s+x\s+)')

def preprocess_batch(texts):
    s = pd.Series(texts).fillna('').astype(str)
    s = s.str.replace(_RE_BIG, '<big_gap>', regex=True)
    s = s.str.replace(_RE_SMALL, '<gap>', regex=True)
    return s.tolist()

print('Preprocessing: minimal (gap normalization only)')

In [ ]:
# ============================================================
# Cell 4: Postprocessing (from reference + fraction fix)
# ============================================================
_PP = {
    'gap': re.compile(r'(\[x\]|\(x\)|\bx\b)', re.I),
    'big_gap': re.compile(r'(\.{3,}|\u2026|\[\.+\])'),
    'annotations': re.compile(r'\((fem|plur|pl|sing|singular|plural|\?|!)\.*\s*\w*\)', re.I),
    'repeated_words': re.compile(r'\b(\w+)(?:\s+\1\b)+'),
    'ws': re.compile(r'\s+'),
    'punct_space': re.compile(r'\s+([.,:])'),
    'repeated_punct': re.compile(r'([.,])\1+'),
}

_SUB_TR = str.maketrans('\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089', '0123456789')
_SPEC_TR = str.maketrans('\u1e2b\u1e2a', 'hH')
_FORBIDDEN = '!?()\'\"\u2014\u2014<>\u2308\u2309\u230a\u230b[]+\u02be/;'
_FORB_TR = str.maketrans('', '', _FORBIDDEN)

_NGRAM_PATS = []
for n in range(4, 1, -1):
    _NGRAM_PATS.append(re.compile(r'\b((?:\w+\s+){' + str(n-1) + r'}\w+)(?:\s+\1\b)+'))

def postprocess_batch(translations):
    s = pd.Series(translations).fillna('').astype(str)
    # Basic cleaning
    s = s.str.translate(_SPEC_TR).str.translate(_SUB_TR)
    s = s.str.replace(_PP['ws'], ' ', regex=True).str.strip()
    # Gaps
    s = s.str.replace(_PP['gap'], '<gap>', regex=True)
    s = s.str.replace(_PP['big_gap'], '<big_gap>', regex=True)
    s = s.str.replace('<gap> <gap>', '<big_gap>', regex=False)
    s = s.str.replace('<big_gap> <big_gap>', '<big_gap>', regex=False)
    # Annotations
    s = s.str.replace(_PP['annotations'], '', regex=True)
    # FRACTIONS FIRST (before '/' removal!)
    s = s.str.replace(r'\b1/3\b', '\u2153', regex=True)
    s = s.str.replace(r'\b2/3\b', '\u2154', regex=True)
    s = s.str.replace(r'\b1/6\b', '\u2159', regex=True)
    s = s.str.replace(r'\b5/6\b', '\u215a', regex=True)
    s = s.str.replace(r'\b1/2\b', '\u00bd', regex=True)
    s = s.str.replace(r'\b1/4\b', '\u00bc', regex=True)
    s = s.str.replace(r'\b3/4\b', '\u00be', regex=True)
    # Decimal fractions
    s = s.str.replace(r'\b0\.5\b', '\u00bd', regex=True)
    s = s.str.replace(r'(\d+)\.5\b', '\\g<1>\u00bd', regex=True)
    s = s.str.replace(r'\b0\.25\b', '\u00bc', regex=True)
    s = s.str.replace(r'(\d+)\.25\b', '\\g<1>\u00bc', regex=True)
    s = s.str.replace(r'\b0\.75\b', '\u00be', regex=True)
    s = s.str.replace(r'(\d+)\.75\b', '\\g<1>\u00be', regex=True)
    # NOW remove forbidden chars (protect gaps)
    s = s.str.replace('<gap>', '\x00GAP\x00', regex=False)
    s = s.str.replace('<big_gap>', '\x00BIG\x00', regex=False)
    s = s.str.translate(_FORB_TR)
    s = s.str.replace('\x00GAP\x00', ' <gap> ', regex=False)
    s = s.str.replace('\x00BIG\x00', ' <big_gap> ', regex=False)
    # Repeated words/n-grams
    s = s.str.replace(_PP['repeated_words'], r'\1', regex=True)
    for pat in _NGRAM_PATS:
        s = s.str.replace(pat, r'\1', regex=True)
    # Punctuation
    s = s.str.replace(_PP['punct_space'], r'\1', regex=True)
    s = s.str.replace(_PP['repeated_punct'], r'\1', regex=True)
    s = s.str.replace(_PP['ws'], ' ', regex=True)
    s = s.str.strip().str.strip('-').str.strip()
    return s.tolist()

# Self-test
_t = postprocess_batch(['He paid 1.5 minas.', '0.5 shekel and 1/3 mina.', '2/3 mina.'])
assert '\u2153' in _t[1], f'1/3 bug: {_t[1]}'
assert '\u2154' in _t[2], f'2/3 bug: {_t[2]}'
print('Postprocessing OK:', _t)

In [ ]:
# ============================================================
# Cell 5: MBR (chrF++ utility — from reference)
# ============================================================
_CHRFPP = sacrebleu.metrics.CHRF(word_order=2)

def _sim(a, b):
    a, b = (a or '').strip(), (b or '').strip()
    if not a or not b: return 0.0
    return float(_CHRFPP.sentence_score(a, [b]).score)

def mbr_pick(cands):
    # Deduplicate
    seen, unique = set(), []
    for c in cands:
        c = str(c).strip()
        if c and c not in seen:
            unique.append(c); seen.add(c)
    if len(unique) == 0: return ''
    if len(unique) == 1: return unique[0]
    # MBR: maximize average similarity to others
    n = len(unique)
    best_i, best_s = 0, -1e9
    for i in range(n):
        s = sum(_sim(unique[i], unique[j]) for j in range(n) if i != j) / max(1, n-1)
        if s > best_s: best_s, best_i = s, i
    return unique[best_i]

print('MBR: chrF++ utility')

In [ ]:
# ============================================================
# Cell 6: Load model (SINGLE GPU, FP32, BetterTransformer)
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Loading: {MODEL_PATH}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model = model.to(device).eval()

# BetterTransformer for 20-50% speedup
try:
    from optimum.bettertransformer import BetterTransformer
    model = BetterTransformer.transform(model)
    print('BetterTransformer: applied')
except Exception as e:
    print(f'BetterTransformer: skipped ({e})')

if device.type == 'cuda':
    try: torch.set_float32_matmul_precision('high')
    except: pass

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded: {n_params/1e6:.0f}M params, dtype={next(model.parameters()).dtype}, device={device}')

In [ ]:
# ============================================================
# Cell 7: Load & preprocess test data
# ============================================================
test_df = pd.read_csv(TEST_PATH)
print(f'Test: {len(test_df)} samples')

raw = test_df['transliteration'].tolist()
processed = preprocess_batch(raw)
test_df['input_text'] = [PREFIX + t for t in processed]

print(f'First 3:')
for _, r in test_df.head(3).iterrows():
    print(f'  [{r["id"]}] {r["input_text"][:80]}')

In [ ]:
# ============================================================
# Cell 8: Dataset + BucketBatchSampler
# ============================================================
class AkkDS(Dataset):
    def __init__(self, ids, texts):
        self.ids, self.texts = ids, texts
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.texts[i]

class BucketSampler(Sampler):
    def __init__(self, dataset, bs, n_buckets=4):
        lengths = [len(t.split()) for _, t in dataset]
        si = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bsz = max(1, len(si) // n_buckets)
        self.batches = []
        for b in range(n_buckets):
            start = b * bsz
            end = None if b == n_buckets - 1 else (b+1) * bsz
            bucket = si[start:end]
            for i in range(0, len(bucket), bs):
                self.batches.append(bucket[i:i+bs])
    def __iter__(self):
        for b in self.batches: yield b
    def __len__(self): return len(self.batches)

def collate(batch):
    ids = [s[0] for s in batch]
    texts = [s[1] for s in batch]
    tok = tokenizer(texts, max_length=MAX_LENGTH, padding=True, truncation=True, return_tensors='pt')
    return ids, tok

ds = AkkDS(test_df['id'].tolist(), test_df['input_text'].tolist())
sampler = BucketSampler(ds, BATCH_SIZE)
loader = DataLoader(ds, batch_sampler=sampler, num_workers=NUM_WORKERS,
                    collate_fn=collate, pin_memory=True,
                    prefetch_factor=2,
                    persistent_workers=NUM_WORKERS > 0)
print(f'DataLoader: {len(loader)} batches')

In [ ]:
# ============================================================
# Cell 9: Inference (MBR — matching reference exactly)
# ============================================================
t0 = time.time()
results = []
checkpoint_freq = 100

gen_common = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    repetition_penalty=REPETITION_PENALTY,
    use_cache=True,
)

with torch.inference_mode():
    for batch_idx, (batch_ids, tokenized) in enumerate(tqdm(loader, desc='Translating')):
        try:
            input_ids = tokenized.input_ids.to(device)
            attn_mask = tokenized.attention_mask.to(device)
            B = input_ids.shape[0]
            
            # Adaptive beams
            seq_len = attn_mask.sum(dim=1).float().mean().item()
            n_beams = max(4, NUM_BEAMS // 2) if seq_len < 100 else NUM_BEAMS
            
            with amp_ctx(USE_BF16):
                if USE_MBR:
                    pools = [[] for _ in range(B)]
                    
                    # Beam candidates
                    nb = max(n_beams, MBR_BEAM_CANDS)
                    beam_out = model.generate(
                        input_ids=input_ids, attention_mask=attn_mask,
                        do_sample=False, num_beams=nb,
                        num_return_sequences=MBR_BEAM_CANDS,
                        length_penalty=LENGTH_PENALTY,
                        early_stopping=True,
                        **gen_common,
                    )
                    beam_txt = tokenizer.batch_decode(beam_out, skip_special_tokens=True)
                    for i in range(B):
                        pools[i].extend(beam_txt[i*MBR_BEAM_CANDS:(i+1)*MBR_BEAM_CANDS])
                    
                    # Sample candidates
                    if MBR_SAMPLE_CANDS > 0:
                        samp_out = model.generate(
                            input_ids=input_ids, attention_mask=attn_mask,
                            do_sample=True, num_beams=1,
                            top_p=MBR_TOP_P, temperature=MBR_TEMPERATURE,
                            num_return_sequences=MBR_SAMPLE_CANDS,
                            **gen_common,
                        )
                        samp_txt = tokenizer.batch_decode(samp_out, skip_special_tokens=True)
                        for i in range(B):
                            pools[i].extend(samp_txt[i*MBR_SAMPLE_CANDS:(i+1)*MBR_SAMPLE_CANDS])
                    
                    # MBR pick
                    for i, bid in enumerate(batch_ids):
                        best = mbr_pick(pools[i])
                        results.append((bid, best))
                else:
                    out = model.generate(
                        input_ids=input_ids, attention_mask=attn_mask,
                        num_beams=n_beams, length_penalty=LENGTH_PENALTY,
                        early_stopping=True, **gen_common,
                    )
                    texts = tokenizer.batch_decode(out, skip_special_tokens=True)
                    for bid, txt in zip(batch_ids, texts):
                        results.append((bid, txt))
            
            # Checkpoint
            if len(results) > 0 and len(results) % checkpoint_freq == 0:
                cp = pd.DataFrame(results, columns=['id', 'translation'])
                cp.to_csv(Path(OUTPUT_DIR) / f'checkpoint_{len(results)}.csv', index=False)
                print(f'  Checkpoint: {len(results)} translations')
            
            if torch.cuda.is_available() and batch_idx % 10 == 0:
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f'Batch {batch_idx} error: {e}')
            for bid in batch_ids:
                results.append((bid, ''))

# Postprocess
raw_trans = [r[1] for r in results]
cleaned = postprocess_batch(raw_trans)
results = [(results[i][0], cleaned[i]) for i in range(len(results))]

elapsed = time.time() - t0
print(f'\nDone: {len(results)} translations in {elapsed:.0f}s ({elapsed/max(len(results),1):.1f}s/each)')

In [ ]:
# ============================================================
# Cell 10: Save submission
# ============================================================
sub = pd.DataFrame(results, columns=['id', 'translation'])
sub = sub.sort_values('id').reset_index(drop=True)
sub['translation'] = sub['translation'].fillna('')

empty = sub['translation'].str.strip().eq('').sum()
lengths = sub['translation'].str.len()
print(f'Total: {len(sub)} translations')
print(f'Empty: {empty} ({empty/max(1,len(sub))*100:.1f}%)')
print(f'Lengths: mean={lengths.mean():.0f}, median={lengths.median():.0f}, min={lengths.min()}, max={lengths.max()}')

print(f'\nSamples:')
for idx in [0, len(sub)//2, len(sub)-1]:
    if idx < len(sub):
        r = sub.iloc[idx]
        print(f'  ID {int(r["id"]):4d}: {str(r["translation"])[:70]}')

sub_path = Path(OUTPUT_DIR) / 'submission.csv'
sub.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path} ({len(sub)} rows)')